# Notebook — Motor de matching y evaluación (Levenshtein, Jaro-Winkler, Fonético, Embeddings)

Este notebook es un **laboratorio de evaluación** para escoger un modelo de matching de nombres.

**Qué hace (alto nivel):**
1. Carga **terceros sintéticos** (CSV) con la etiqueta `coincidencia` (ground truth).
2. Carga **consolidado** (SQLite) como base de búsqueda.
3. Normaliza texto para que las comparaciones sean consistentes.
4. Genera candidatos (blocking) para acelerar el matching.
5. Calcula scores para 4 enfoques (Levenshtein, Jaro-Winkler, Fonético, Embeddings).
6. Evalúa con **Precision / Recall / F1** en varios umbrales.
7. Permite inspeccionar errores (FP/FN) para mejorar y decidir.

> Tip: Primero corre con `df_sample` (muestra) para iterar rápido. Cuando estés listo, aumenta el tamaño o corre todo.


In [1]:
# ============================================
# 1) IMPORTS
# --------------------------------------------
# Este notebook compara varios "modelos" de matching de nombres:
# - Levenshtein
# - Jaro-Winkler
# - Fonético (Double Metaphone)
# - Embeddings (sentence-transformers + cosine similarity)
#
# Además:
# - Carga datos desde CSV (terceros sintéticos) y SQLite (consolidado)
# - Normaliza texto para que la comparación sea consistente
# - Genera candidatos (blocking) para acelerar el matching
# - Evalúa con Precision / Recall / F1 a distintos umbrales
# ============================================

import os               # Variables de entorno (por si DB_PATH existe)
import re               # Normalización de texto con regex
import sqlite3          # Conexión a SQLite
from pathlib import Path

import numpy as np      # Arrays y operaciones numéricas
import pandas as pd     # DataFrames

# RapidFuzz: implementaciones rápidas de distancias/similitudes
from rapidfuzz.distance import Levenshtein, JaroWinkler

# Metaphone: algoritmo fonético (útil para errores por pronunciación/ortografía)
from metaphone import doublemetaphone

# Embeddings: representación semántica del nombre
from sentence_transformers import SentenceTransformer

# Métricas de clasificación (score -> predicción por umbral)
from sklearn.metrics import precision_score, recall_score, f1_score

# Similaridad coseno para embeddings
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\pc\OneDrive\Documentos\Data project\tusdatos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuración de rutas
Ajusta estas rutas si tu proyecto usa otra estructura.


In [2]:
# ============================================
# 2) CONFIGURACIÓN DE RUTAS Y FUENTES
# --------------------------------------------
# - DB_FILE: base SQLite donde está la tabla "consolidado"
# - TERCEROS_CSV: CSV generado con tu script (terceros_sinteticos.csv)
#
# Nota:
# Path(".").resolve().parents[1] asume que el notebook está en una carpeta
# tipo: <repo>/notebooks/<este_notebook>.ipynb
# Si mueves el notebook, ajusta parents[...] o usa una ruta absoluta.
# ============================================

PROJECT_ROOT = Path(".").resolve().parents[1]     # Raíz del repo (ajusta si cambia tu estructura)
DB_FILE = PROJECT_ROOT / "analytics.db"           # Archivo SQLite
CONSOLIDADO_TABLE = "consolidado"                 # Tabla base "real"
TERCEROS_CSV = PROJECT_ROOT / "data" / "terceros_sinteticos.csv"  # Sintéticos

print("DB_FILE:", DB_FILE)
print("SYNTH_CSV:", TERCEROS_CSV)


DB_FILE: C:\Users\pc\OneDrive\Documentos\Data project\tusdatos\analytics.db
SYNTH_CSV: C:\Users\pc\OneDrive\Documentos\Data project\tusdatos\data\terceros_sinteticos.csv


In [3]:
# ============================================
# 3) (OPCIONAL) GENERAR BASE SINTÉTICA
# --------------------------------------------
# Este comando ejecuta tu generador de terceros sintéticos.
# Útil para que el notebook sea reproducible.
#
# Si YA tienes el CSV y no quieres regenerarlo cada vez,
# comenta esta celda (poniendo # al inicio de la línea) o bórrala.
# ============================================

# Crear base de terceros si no existe / si quieres regenerarla
# !python -m pipeline.matching.generate_terceros


In [4]:
# ============================================
# 4) CARGA DE DATAFRAMES (SINTÉTICOS + CONSOLIDADO)
# --------------------------------------------
# df_terceros: dataset sintético con label "coincidencia" (1/0)
# df_cons: base consolidada (contra la que hacemos matching)
#
# Buenas prácticas:
# - dtype=str para evitar que documentos se vuelvan numéricos
# - fillna("") para no romper funciones por None/NaN
# ============================================

df_terceros = pd.read_csv(TERCEROS_CSV, dtype=str).fillna("")
print("terceros:", df_terceros.shape)

# Abrimos conexión a SQLite y leemos consolidado
conn = sqlite3.connect(DB_FILE)

# Importante: traemos solo columnas necesarias para el matching (ahorra memoria)
df_cons = pd.read_sql_query(
    f"""
    SELECT
        tipo_sujeto, nombres, apellidos, fecha_nacimiento, nacionalidad,
        numero_documento, aliases
    FROM {CONSOLIDADO_TABLE}
    """,
    conn
).fillna("")

conn.close()
print("consolidado:", df_cons.shape)

# Vista rápida de terceros
df_terceros.head(2)


terceros: (10000, 10)
consolidado: (23608, 7)


,id_tercero,tipo_sujeto,nombres,apellidos,fecha_nacimiento,nacionalidad,numero_documento,tipo_documento,pais_residencia,coincidencia
0,T0000000,PERSONA_NATURAL,JUAN SARA,HERNANDEZ MARTINEZ,1964-03-24,VENEZOLANA,X2458591,PASAPORTE,ESPAÑA,0
1,T0000001,PERSONA_NATURAL,ANDRES LUISA,MARTINEZ TORRES,1988-01-18,PERUANA,U8038374,PASAPORTE,PERU,0


In [5]:
# ============================================
# 5) NORMALIZACIÓN + FEATURES BASE (nombre_full / doc_norm)
# --------------------------------------------
# Objetivo:
# - Convertir texto a una forma comparable entre datasets.
# - Evitar diferencias por: mayúsculas/minúsculas, tildes/símbolos, espacios dobles, etc.
#
# Outputs:
# - nombre_full: "NOMBRES APELLIDOS" normalizado (string)
# - doc_norm: documento normalizado (string)
# ============================================

def norm_text(s: str) -> str:
    # Manejo defensivo: NaN/None -> ""
    s = "" if s is None else str(s)

    # Estandarizamos a MAYÚSCULAS y sin espacios al inicio/fin
    s = s.upper().strip()

    # Dejamos solo letras, números y espacios (todo lo demás -> espacio)
    # Ej: "PÉREZ-LÓPEZ" -> "P REZ L PEZ" (si luego quieres preservar tildes, se puede mejorar)
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)

    # Colapsamos espacios múltiples a uno solo
    s = re.sub(r"\s+", " ", s).strip()
    return s


def full_name(nombres: str, apellidos: str) -> str:
    # Normalizamos por separado y luego concatenamos
    n = norm_text(nombres)
    a = norm_text(apellidos)
    return (n + " " + a).strip()


# Creamos el nombre unificado en ambos dataframes
# (esto reduce complejidad: el modelo siempre compara nombre_full vs nombre_full)
df_terceros["nombre_full"] = df_terceros.apply(
    lambda r: full_name(r["nombres"], r["apellidos"]),
    axis=1
)
df_cons["nombre_full"] = df_cons.apply(
    lambda r: full_name(r["nombres"], r["apellidos"]),
    axis=1
)

# Normalizamos documento para poder usarlo en blocking (y en versiones futuras, como feature fuerte)
df_terceros["doc_norm"] = df_terceros["numero_documento"].apply(norm_text)
df_cons["doc_norm"] = df_cons["numero_documento"].apply(norm_text)

# Vista rápida de columnas claves para el experimento
df_terceros[["id_tercero", "nombre_full", "doc_norm", "coincidencia"]].head(5)


,id_tercero,nombre_full,doc_norm,coincidencia
0,T0000000,JUAN SARA HERNANDEZ MARTINEZ,X2458591,0
1,T0000001,ANDRES LUISA MARTINEZ TORRES,U8038374,0
2,T0000002,CARLOS JUAN RODRIGUEZ GARCIA,229494902,0
3,T0000003,ANA ANA RAMIREZ HERNANDEZ,D7350753,0
4,T0000004,DAVID CAMILA LOPEZ RAMIREZ,H5855124,0


In [6]:
# ============================================
# 6) CANDIDATE GENERATION (BLOCKING)
# --------------------------------------------
# Problema:
# Si comparas cada tercero contra TODO consolidado, el costo es:
#   O(N_terceros * N_consolidado)
# Eso explota rápido.
#
# Solución:
# "Blocking": generar un conjunto pequeño de candidatos probables.
# Aquí usamos dos estrategias simples:
#  1) Si hay doc_norm y existe en consolidado -> candidatos = todos con ese doc.
#  2) Si no hay doc -> candidatos por prefijo del nombre (primeros BLOCK_LEN chars).
#
# MAX_CANDIDATES:
# Evita bloques gigantes (ej: prefijos comunes como "MAR").
# ============================================

BLOCK_LEN = 3
MAX_CANDIDATES = 2000

# -------------------------
# 6.1) Índice por documento exacto
# doc_index: { "123456": [idx1, idx2, ...] }
# -------------------------
doc_index = {}
for idx, d in enumerate(df_cons["doc_norm"].values):
    if d:  # ignoramos vacíos
        # setdefault crea lista si no existe; append agrega el índice
        doc_index.setdefault(d, []).append(idx)

# -------------------------
# 6.2) Índice por bloque de nombre (prefijo)
# name_block_index: { "JUA": [idxs...], "CAR": [idxs...], ... }
# -------------------------
name_block_index = {}
for idx, n in enumerate(df_cons["nombre_full"].values):
    key = n[:BLOCK_LEN] if len(n) >= BLOCK_LEN else n
    name_block_index.setdefault(key, []).append(idx)

# -------------------------
# 6.3) Función para obtener candidatos de un tercero
# -------------------------
def get_candidate_indices(nombre_full: str, doc_norm: str) -> np.ndarray:
    # (1) Si hay doc y existe en el índice, usamos doc como blocking fuerte
    if doc_norm and doc_norm in doc_index:
        return np.array(doc_index[doc_norm], dtype=np.int32)

    # (2) Si no hay doc útil, hacemos blocking por prefijo de nombre
    key = nombre_full[:BLOCK_LEN] if len(nombre_full) >= BLOCK_LEN else nombre_full
    cands = name_block_index.get(key, [])

    # Recortamos por seguridad si el bloque es gigante
    if len(cands) > MAX_CANDIDATES:
        cands = cands[:MAX_CANDIDATES]

    return np.array(cands, dtype=np.int32)


In [7]:
# ============================================
# 7) MODELOS / SCORERS
# --------------------------------------------
# Cada función devuelve un score en [0,1] (idealmente):
# - Levenshtein / JaroWinkler: score continuo (0..1)
# - Fonético: score discreto (0 o 1)
# - Embeddings: cosine similarity (~0..1)
#
# NOTA:
# Para embeddings, el modelo se carga una sola vez (embedder),
# porque cargarlo dentro de la función sería MUY lento.
# ============================================

def score_lev(a: str, b: str) -> float:
    # Levenshtein normalizado: 1 = idéntico, 0 = completamente distinto
    if not a or not b:
        return 0.0
    return float(Levenshtein.normalized_similarity(a, b))


def score_jw(a: str, b: str) -> float:
    # Jaro-Winkler: favorece coincidencias con mismo prefijo (útil para nombres)
    if not a or not b:
        return 0.0
    return float(JaroWinkler.normalized_similarity(a, b))


def score_phon(a: str, b: str) -> float:
    # Double Metaphone: compara "sonido" aproximado.
    # Útil si "GONZALES" vs "GONZALEZ", etc.
    if not a or not b:
        return 0.0
    pa = doublemetaphone(a)[0]
    pb = doublemetaphone(b)[0]
    return 1.0 if (pa and pa == pb) else 0.0


# Cargamos el modelo de embeddings UNA sola vez
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def score_emb(a: str, b: str) -> float:
    # Embeddings: convierte strings a vectores y mide similitud coseno
    if not a or not b:
        return 0.0

    # encode devuelve embeddings (vectores) para cada string
    emb = embedder.encode([a, b])

    # cosine_similarity devuelve matriz 1x1; extraemos el valor
    return float(cosine_similarity([emb[0]], [emb[1]])[0][0])


# Registro de modelos para poder iterar fácilmente
MODELS = {
    "levenshtein": score_lev,
    "jaro_winkler": score_jw,
    "phonetic": score_phon,
    "embeddings": score_emb,
}


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 321.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# ============================================
# 8) RUNNER DE MATCHING: TOP-1 POR REGISTRO
# --------------------------------------------
# Para cada tercero (left):
#  1) Genera candidatos (blocking)
#  2) Calcula score vs cada candidato
#  3) Se queda con el mejor (Top-1)
#
# Output:
# - best_score: score del mejor candidato
# - best_name_right: nombre del candidato ganador
# - best_doc_match: 1 si el documento coincidió exactamente (señal auxiliar)
# - label: tu ground truth (coincidencia: 1/0)
#
# Importante:
# Este experimento mide: "¿separa bien positivos vs negativos por score?"
# No mide (aún) si encontró el REGISTRO exacto, porque el sintético no guarda el id origen.
# ============================================

def run_top1(df_left: pd.DataFrame, model_name: str) -> pd.DataFrame:
    scorer = MODELS[model_name]  # función de score que vamos a usar
    out = []                     # aquí acumulamos resultados por cada tercero

    for _, r in df_left.iterrows():
        # ----- datos del tercero (left) -----
        id_left = r["id_tercero"]
        nombre_left = r["nombre_full"]
        doc_left = r["doc_norm"]
        label = int(r["coincidencia"])  # 1 = debería matchear con algo en consolidado

        # ----- candidatos (right) -----
        cand_idx = get_candidate_indices(nombre_left, doc_left)

        # Si no hay candidatos, registramos un resultado "vacío"
        if cand_idx.size == 0:
            out.append({
                "id_left": id_left,
                "label": label,
                "best_score": 0.0,
                "best_doc_match": 0,
                "best_name_right": "",
            })
            continue

        # Subset de consolidado solo con candidatos
        cands = df_cons.iloc[cand_idx]

        # ----- búsqueda del mejor candidato -----
        best_score = -1.0
        best_name = ""
        best_doc_match = 0

        # Recorremos candidatos y tomamos el mayor score
        for _, rr in cands.iterrows():
            nombre_right = rr["nombre_full"]

            # score del modelo para este par (left vs right)
            s = scorer(nombre_left, nombre_right)

            # si mejora el score, actualizamos el "ganador"
            if s > best_score:
                best_score = s
                best_name = nombre_right

                # señal auxiliar: ¿el documento coincide exacto?
                best_doc_match = 1 if (doc_left and doc_left == rr["doc_norm"]) else 0

        # Guardamos resultado final del tercero
        out.append({
            "id_left": id_left,
            "label": label,
            "best_score": float(best_score),
            "best_doc_match": int(best_doc_match),
            "name_left": nombre_left,
            "best_name_right": best_name,
        })

    return pd.DataFrame(out)


# Ejemplo rápido: correr con una muestra chica para verificar que funciona
res_lev = run_top1(df_terceros.sample(200, random_state=42), "levenshtein")
res_lev.head()


,id_left,label,best_score,best_doc_match,name_left,best_name_right
0,T0006252,0,0.320000,0,ANA JORGE GOMEZ GOMEZ,ANALOG TECHNOLOGY LIMITED
1,T0004684,0,0.500000,0,ANDRES CAMILA MARTINEZ LOPEZ,ANDREA MARTINA LIMITED
2,T0001731,0,0.269231,0,LOPEZ LTDA,LOPEZ JOSE FRANCISCO LOPEZ
3,T0004742,0,0.366667,0,CAMILA CARLOS SANCHEZ MARTINEZ,CAMELIAS BAR S A DE C V
4,T0004521,0,0.302326,0,CAMILA JUAN LOPEZ MARTINEZ,CAMPOS TIRADO LINDA ELIZABETH CAMPOS TIRADO


In [9]:
# ============================================
# 9) EVALUACIÓN (Precision / Recall / F1) POR UMBRAL
# --------------------------------------------
# Los modelos devuelven un score continuo. Para decidir "MATCH / NO MATCH"
# necesitamos un umbral:
#   pred = 1 si score >= threshold
#
# Métricas:
# - Precision: de los predichos como match, cuántos eran verdaderos
# - Recall: de los verdaderos match, cuántos detectamos
# - F1: balance entre precision y recall
#
# Nota de negocio (listas de sanciones):
# - Suele priorizarse recall alto (no perder matches reales),
#   pero sin disparar demasiados falsos positivos.
# ============================================

def eval_threshold(df_res: pd.DataFrame, threshold: float) -> dict:
    # y_true: etiqueta real (coincidencia)
    y_true = df_res["label"].values

    # y_pred: decisión por umbral
    y_pred = (df_res["best_score"].values >= threshold).astype(int)

    # zero_division=0 evita warnings cuando no hay predicciones positivas
    return {
        "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "positives": int(y_true.sum()),
        "predicted_positive": int(y_pred.sum()),
    }


def evaluate_model(df_left: pd.DataFrame, model_name: str, thresholds=(0.80, 0.85, 0.90, 0.93)):
    # 1) corremos matching top-1
    df_res = run_top1(df_left, model_name)

    # 2) evaluamos a varios thresholds
    rows = [eval_threshold(df_res, t) for t in thresholds]

    # devolvemos el resumen (tabla) y el detalle por registro
    return pd.DataFrame(rows), df_res


# Corre en una muestra primero (para iterar rápido mientras ajustas)
df_sample = df_terceros.sample(1000, random_state=42)

summary_lev, res_lev = evaluate_model(df_sample, "levenshtein")
summary_jw,  res_jw  = evaluate_model(df_sample, "jaro_winkler")
summary_ph,  res_ph  = evaluate_model(df_sample, "phonetic")
summary_emb, res_emb = evaluate_model(df_sample, "embeddings")

# Muestra el resumen de un modelo (puedes cambiarlo por summary_jw, etc.)
summary_lev


,threshold,precision,recall,f1,positives,predicted_positive
0,0.80,1.0,0.403226,0.574713,62,25
1,0.85,1.0,0.387097,0.558140,62,24
2,0.90,1.0,0.387097,0.558140,62,24
3,0.93,1.0,0.370968,0.541176,62,23


In [10]:
# ============================================
# 10) COMPARATIVO FINAL DE MODELOS
# --------------------------------------------
# Unimos los resúmenes de todos los modelos en una tabla para comparar.
# Ordenamos por threshold y luego por F1 para ver "quién gana" en cada umbral.
# ============================================

def add_model_col(df_summary: pd.DataFrame, model: str) -> pd.DataFrame:
    # Insertamos columna "model" al inicio para identificar el origen del resumen
    df2 = df_summary.copy()
    df2.insert(0, "model", model)
    return df2


comparativo = pd.concat([
    add_model_col(summary_lev, "levenshtein"),
    add_model_col(summary_jw,  "jaro_winkler"),
    add_model_col(summary_ph,  "phonetic"),
    add_model_col(summary_emb, "embeddings"),
], ignore_index=True)

# Ordenamos para ver fácilmente el mejor F1 por umbral
comparativo.sort_values(["threshold", "f1"], ascending=[True, False])


,model,threshold,precision,recall,f1,positives,predicted_positive
12,embeddings,0.80,0.738095,0.500000,0.596154,62,42
0,levenshtein,0.80,1.000000,0.403226,0.574713,62,25
8,phonetic,0.80,1.000000,0.306452,0.469136,62,19
4,jaro_winkler,0.80,0.090016,0.887097,0.163447,62,611
1,levenshtein,0.85,1.000000,0.387097,0.558140,62,24
13,embeddings,0.85,0.764706,0.419355,0.541667,62,34
9,phonetic,0.85,1.000000,0.306452,0.469136,62,19
5,jaro_winkler,0.85,0.222798,0.693548,0.337255,62,193
6,jaro_winkler,0.90,1.000000,0.500000,0.666667,62,31
2,levenshtein,0.90,1.000000,0.387097,0.558140,62,24


In [11]:
# ============================================
# 11) ANÁLISIS DE ERRORES (Falsos Positivos / Falsos Negativos)
# --------------------------------------------
# Esto es lo que más valor te da para mejorar:
# - FP: "dije match" pero era negativo (coincidencia=0) -> ruido operativo
# - FN: "dije no match" pero era positivo (coincidencia=1) -> riesgo alto en sanciones
#
# Recomendación:
# - Empieza revisando FN: suelen decirte qué patrón no está cubriendo el modelo
#   (typos, alias, parciales, etc.)
# ============================================

def inspect_false_positives(df_res: pd.DataFrame, threshold: float, n=10):
    # FP: label=0 pero score >= threshold
    fp = df_res[(df_res["label"] == 0) & (df_res["best_score"] >= threshold)].copy()
    return fp.sort_values("best_score", ascending=False).head(n)


def inspect_false_negatives(df_res: pd.DataFrame, threshold: float, n=10):
    # FN: label=1 pero score < threshold
    fn = df_res[(df_res["label"] == 1) & (df_res["best_score"] < threshold)].copy()
    return fn.sort_values("best_score", ascending=True).head(n)


# Ejemplo: ver falsos negativos de Levenshtein a threshold 0.90
inspect_false_negatives(res_lev, 0.90, n=10)


,id_left,label,best_score,best_doc_match,name_left,best_name_right
545,T0009362,1,0.000000,0,NaN,
284,T0009445,1,0.000000,0,NaN,
602,T0009765,1,0.000000,0,NaN,
991,T0009965,1,0.114286,1,ADEL,ADEL BEN AL AZHAR BEN YOUSSEF HAMDI
730,T0009975,1,0.147059,1,SALEM,SALEM NOR ELDIN AMOHAMED AL DABSKI
25,T0009920,1,0.147059,1,HASAN,HASAN AL SALAHAYN SALIH AL SHA ARI
654,T0009873,1,0.157895,1,ABU,ABU ZAYD UMAR DORDA
533,T0009919,1,0.161290,1,ABDUL,ABDUL QADEER BASIR ABDUL BASEER
17,T0009930,1,0.181818,1,KO,KO CHOL MAN
20,T0009896,1,0.181818,1,JO,JO YONG WON
